<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L06-review-queues-real-reviewers/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L06-review-queues-real-reviewers/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/document-intelligence/lessons/P02-L06-review-queues-real-reviewers/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/document-intelligence/lessons/P02-L06-review-queues-real-reviewers/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P02-L06 · Review queues that survive contact with humans

**You will build:** the instruments a human review team is measured with — raw agreement and
Cohen's kappa between two reviewers, an SLA-aware queue that serves the most urgent work first
and counts what it breaches, a two-tier escalation policy, and the ceiling your reviewers' own
agreement puts on every score your harness reports.

**Time:** ~70 minutes · **Runs on:** a laptop CPU, no GPU, no download, no model API ·
**Prerequisites:** T00-L01 (the 8 GB track), P02-L01 (field extraction you can measure).

Module 1's `apply_reviews` gave every routed cell its gold value. That reviewer was perfect:
never tired, never rushed, never in disagreement with a colleague. This lesson replaces them
with people. They disagree with each other, they get worse as a shift goes on, and the queue
they work has deadlines.

By the end you will be able to:

1. Implement raw agreement and Cohen's kappa over the labels either reviewer used, including
   kappa's edge cases: perfect agreement, chance level, systematic disagreement, one shared
   label, and a label one reviewer never uses.
2. Implement an SLA-aware queue that serves arrived work earliest deadline first, and measure
   its breach rate against first-in-first-out and module 1's confidence order.
3. Implement a two-tier escalation policy and measure the tier-1 mistakes it sends to a senior
   reviewer against a random escalation of the same size.
4. Measure throughput against quality across shift lengths, and choose one against a stated
   breach-rate target.
5. Compute, from your own kappa, the ceiling your harness can see, check it against the
   simulation's truth, and explain why it is not 1.0.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import random
import re
import sys
import time
import traceback
from typing import Callable, Iterable, Mapping, NamedTuple, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)
print("no model API, no labelling tool, no real reviewers — a simulated team, from a fixed seed,")
print("whose every mistake is recorded, so that your measurements can be checked against the truth.\n")

# Module 1's schema: six fields, and the TYPE of each one, because field type is the unit of
# policy — in this lesson it also decides a review deadline and whether a change needs a
# second pair of eyes.
SCHEMA: dict[str, str] = {
    "invoice_id": "id",
    "invoice_date": "date",
    "total_amount": "money",
    "currency": "id",
    "counterparty": "text",
    "payment_terms_days": "integer",
}
FIELD_INDEX = {field: i for i, field in enumerate(SCHEMA)}

# Module 1's five cell labels. Every (document, field) cell lands in exactly one of them.
ERROR_LABELS = ("correct", "miss", "spurious", "wrong_value", "true_negative")

# Module 1's matching modes, minus fuzzy: module 1 measured what fuzzy costs on a money field.
MATCH_MODES = ("exact", "normalised")

MONTH_NAMES = ("January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December")
MONTH_INDEX = {name.lower(): i + 1 for i, name in enumerate(MONTH_NAMES)}

COMPANY_SUFFIXES = frozenset({
    "gmbh", "bv", "nv", "ag", "kg", "ltd", "limited", "inc", "incorporated",
    "plc", "llc", "sa", "sas", "srl", "spa", "oy", "ab", "as", "co",
})


class FieldScore(NamedTuple):
    """What one field scored over the whole corpus, under one matching mode (module 1)."""
    field: str
    mode: str
    tp: int
    fp: int
    fn: int
    precision: float
    recall: float
    f1: float


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("agreement_table", "raw_agreement"),
    "exercise 2": ("chance_agreement", "cohens_kappa"),
    "exercise 3": ("sla_schedule", "breach_report"),
    "exercise 4": ("escalate",),
    "exercise 5": ("harness_ceiling",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 1"] -> "exercise 1 (agreement_table, raw_agreement)"; several -> "exercises 1 and 2"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. Module 1's records, carried over

This notebook stands alone, so module 1's corpus and extractor are reproduced here unchanged:
the same remittance advices from the same seed, and the same mediocre extractor with the same
seeded mistakes. Run it; nothing in this cell is yours to change.

One word changes meaning. Module 1 called the generator's value `gold` and trusted it as what
an annotator typed. Here the simulation keeps it as the **truth**: the value really printed on
the page. The key is still `gold`, so module 1's functions run untouched — and section 10
shows what a human labeller does to it.

In [ ]:
_COMPANIES = (
    "Nordwind Logistik GmbH", "Vantor Marine B.V.", "Helix Pharma Limited",
    "Caldera Energy PLC", "Brightwater Analytics Ltd", "Orsini Costruzioni SRL",
    "Kestrel Freight Inc", "Aalto Terveys Oy", "Meridian Custody AG",
    "Sable & Finch LLP", "Dunbar Reinsurance Ltd", "Petrarca Chimica SpA",
    "Lindqvist Verkstad AB", "Hollandse Kaasunie N.V.", "Argent Clearing SA",
    "Torrent Robotics Inc", "Vesper Maritime AS", "Kaneko Precision KK",
)
_CURRENCIES = {"EUR": "€", "USD": "$", "GBP": "£"}


def _surface_date(year: int, month: int, day: int, style: int) -> str:
    """The same date as an annotator would have found it printed on the page."""
    if style == 0:
        return f"{year:04d}-{month:02d}-{day:02d}"
    if style == 1:
        return f"{day:02d}/{month:02d}/{year:04d}"       # day first: the corpus is European
    return f"{day} {MONTH_NAMES[month - 1]} {year}"


def _surface_money(cents: int, currency: str, style: int) -> str:
    whole, part = divmod(cents, 100)
    if style == 0:
        return f"{_CURRENCIES[currency]}{whole:,}.{part:02d}"
    if style == 1:
        return f"{whole}.{part:02d} {currency}"
    return f"{currency} {whole:,}.{part:02d}"


def _surface_id(year: int, serial: int, style: int) -> str:
    if style == 0:
        return f"INV-{year}-{serial:04d}"
    if style == 1:
        return f"INV/{year}/{serial:04d}"
    return f"Inv {year} {serial:04d}"


def build_corpus(n_docs: int = 180, seed: int = 20260916) -> list[dict]:
    """Module 1's corpus, unchanged: deterministic documents with the values on each page."""
    rng = random.Random(seed)
    corpus = []
    for i in range(n_docs):
        year = rng.choice((2025, 2026))
        month, day, serial = rng.randint(1, 12), rng.randint(1, 28), rng.randint(1, 9999)
        currency = rng.choice(tuple(_CURRENCIES))
        cents = rng.randint(1_50, 480_000_00)
        company = rng.choice(_COMPANIES)
        terms = rng.choice(("14", "30", "30", "45", "60", "", ""))  # 2 in 7 say nothing
        d_style, m_style, i_style = rng.randrange(3), rng.randrange(3), rng.randrange(3)
        gold = {
            "invoice_id": _surface_id(year, serial, i_style),
            "invoice_date": _surface_date(year, month, day, d_style),
            "total_amount": _surface_money(cents, currency, m_style),
            "currency": currency,
            "counterparty": company,
            "payment_terms_days": terms,
        }
        terms_line = f"Payment terms: net {terms} days\n" if terms else ""
        text = (
            f"REMITTANCE ADVICE\n"
            f"Invoice {gold['invoice_id']}\n"
            f"Issued {gold['invoice_date']}\n"
            f"Supplier: {company}\n"
            f"{terms_line}"
            f"Amount due: {gold['total_amount']}\n"
        )
        corpus.append({"doc_id": f"DOC-{i:04d}", "text": text, "gold": gold})
    return corpus


def _reformat(value: str, field_type: str) -> str:
    """The extractor's house style: same meaning, different surface."""
    if field_type == "date":
        m = re.match(r"^(\d{4})-(\d{2})-(\d{2})$", value)
        if m:
            return f"{int(m.group(3))}/{int(m.group(2))}/{m.group(1)}"
        m = re.match(r"^(\d{2})/(\d{2})/(\d{4})$", value)
        if m:
            return f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
        m = re.match(r"^(\d{1,2}) (\w+) (\d{4})$", value)
        if m:
            return f"{m.group(3)}-{MONTH_INDEX[m.group(2).lower()]:02d}-{int(m.group(1)):02d}"
        return value
    if field_type == "money":
        digits = re.sub(r"[^\d.]", "", value.replace(",", ""))
        return digits
    if field_type == "id":
        return re.sub(r"[^A-Za-z0-9]", "", value).upper()
    if field_type == "text":
        return value.replace(" B.V.", "").replace(" GmbH", "").replace(" Ltd", "")
    if field_type == "integer":
        return f"net {value}" if value else value
    return value


def _ocr_noise(value: str, rng: random.Random) -> str:
    """One substituted character, the way a scan smudges a letter."""
    letters = [i for i, c in enumerate(value) if c.isalpha()]
    if not letters:
        return value
    i = rng.choice(letters)
    chars = list(value)
    chars[i] = rng.choice("aeoirnmcl")
    return "".join(chars)


def _corrupt(value: str, field_type: str, rng: random.Random) -> str:
    """A genuinely wrong answer: same shape, different meaning."""
    if field_type == "money":
        digits = [i for i, c in enumerate(value) if c.isdigit()]
        if digits:
            i = rng.choice(digits)
            chars = list(value)
            chars[i] = rng.choice([d for d in "0123456789" if d != chars[i]])
            return "".join(chars)
        return value + "1"
    if field_type in {"integer", "id"}:
        digits = [i for i, c in enumerate(value) if c.isdigit()]
        if len(digits) >= 2:
            i, j = rng.sample(digits, 2)
            chars = list(value)
            chars[i], chars[j] = chars[j], chars[i]
            if "".join(chars) != value:
                return "".join(chars)
        return value + "1"
    if field_type == "date":
        m = re.search(r"\b(\d{1,2})\b", value)
        if m:
            bumped = str((int(m.group(1)) % 28) + 1).zfill(len(m.group(1)))
            return value[:m.start(1)] + bumped + value[m.end(1):]
        return value
    return rng.choice([c for c in _COMPANIES if c != value])


def run_extractor(corpus: Sequence[Mapping], seed: int = 7) -> list[dict]:
    """Module 1's stand-in extractor, unchanged: predictions and confidences for every cell."""
    records = []
    for doc in corpus:
        rng = random.Random(seed * 100_003 + int(doc["doc_id"][4:]))
        pred, conf = {}, {}
        for field, field_type in SCHEMA.items():
            truth = doc["gold"][field]
            roll = rng.random()
            if not truth:
                if roll < 0.22:                       # invents terms that were never printed
                    pred[field] = rng.choice(("30", "14", "60"))
                    conf[field] = round(rng.uniform(0.30, 0.72), 3)
                else:
                    pred[field], conf[field] = "", round(rng.uniform(0.80, 0.98), 3)
                continue
            if roll < 0.09:                           # found nothing
                pred[field], conf[field] = "", 0.0
            elif roll < 0.16:                         # genuinely wrong value
                pred[field] = _corrupt(truth, field_type, rng)
                conf[field] = round(rng.uniform(0.34, 0.86), 3)
            elif roll < 0.23 and field_type == "text":  # smudged scan
                pred[field] = _ocr_noise(truth, rng)
                conf[field] = round(rng.uniform(0.40, 0.78), 3)
            elif roll < 0.72:                         # right answer, house-style surface
                pred[field] = _reformat(truth, field_type)
                conf[field] = round(rng.uniform(0.62, 0.95), 3)
            else:                                     # right answer, verbatim
                pred[field] = truth
                conf[field] = round(rng.uniform(0.86, 0.99), 3)
        records.append({"doc_id": doc["doc_id"], "gold": dict(doc["gold"]),
                        "pred": pred, "conf": conf})
    return records


CORPUS = build_corpus()
RECORDS = run_extractor(CORPUS)
BY_ID = {record["doc_id"]: record for record in RECORDS}
print(f"{len(RECORDS)} documents, {len(SCHEMA)} fields each = "
      f"{len(RECORDS) * len(SCHEMA)} (document, field) cells")
print("one record, in module 1's review-queue format:")
print({key: RECORDS[3][key] for key in ("doc_id", "pred", "conf")})

The harness, carried over the same way: module 1's normaliser, matcher, per-field scorer
(a wrong value is a false positive AND a false negative), unweighted macro F1, cell labels,
confidence-ordered `review_queue` with its `(doc_id, field)` tie-break, and `apply_reviews`
with its perfect reviewer. The last line of output is module 1's promise, re-measured: what a
reviewer who never errs buys at this lesson's review budget.

In [ ]:
def normalise_value(value: str, field_type: str) -> str:
    """Module 1's normaliser, unchanged."""
    if not isinstance(value, str) or not value.strip():
        return ""
    if field_type not in set(SCHEMA.values()):
        raise ValueError(f"unknown field_type {field_type!r}; expected one of "
                         f"{sorted(set(SCHEMA.values()))}")
    raw = " ".join(value.split())
    fallback = raw.upper()

    if field_type == "money":
        body = re.sub(r"[^0-9.,-]", "", raw)
        last_comma, last_dot = body.rfind(","), body.rfind(".")
        if last_comma >= 0 and last_dot >= 0:
            if last_comma > last_dot:                       # 1.234,50 — decimal comma
                body = body.replace(".", "").replace(",", ".")
            else:                                           # 1,234.50 — decimal dot
                body = body.replace(",", "")
        elif last_comma >= 0:
            body = body.replace(",", ".") if re.search(r",\d{1,2}$", body) \
                else body.replace(",", "")
        try:
            return f"{float(body):.2f}"
        except ValueError:
            return fallback

    if field_type == "date":
        m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})$", raw)
        if m:
            y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        else:
            m = re.match(r"^(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})$", raw)
            if m:
                d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
            else:
                m = re.match(r"^(\d{1,2})\s+([A-Za-z]+),?\s+(\d{4})$", raw)
                if m and m.group(2).lower() in MONTH_INDEX:
                    d, mo, y = int(m.group(1)), MONTH_INDEX[m.group(2).lower()], int(m.group(3))
                else:
                    return fallback
        if not (1 <= mo <= 12 and 1 <= d <= 31):
            return fallback
        return f"{y:04d}-{mo:02d}-{d:02d}"

    if field_type == "integer":
        m = re.search(r"\d+", raw)
        return str(int(m.group(0))) if m else fallback

    if field_type == "id":
        stripped = re.sub(r"[^A-Za-z0-9]", "", raw)
        return stripped.upper() if stripped else fallback

    lowered = raw.lower().replace(".", "").replace(",", "")
    tokens = re.sub(r"[^a-z0-9]+", " ", lowered).split()
    while tokens and tokens[-1] in COMPANY_SUFFIXES:
        tokens.pop()
    return " ".join(tokens)


def match_value(pred: str, gold: str, field_type: str, mode: str) -> bool:
    """Module 1's matcher, minus the fuzzy mode module 1 talked you out of."""
    if mode not in MATCH_MODES:
        raise ValueError(f"unknown mode {mode!r}; expected one of {list(MATCH_MODES)}")
    if not isinstance(pred, str) or not isinstance(gold, str):
        return False
    if not pred.strip() or not gold.strip():
        return False
    if mode == "exact":
        return pred == gold
    np_, ng_ = normalise_value(pred, field_type), normalise_value(gold, field_type)
    return bool(np_) and bool(ng_) and np_ == ng_


def score_field(records: Sequence[Mapping], field: str, mode: str) -> FieldScore:
    """Module 1's scorer, unchanged: a wrong value is a false positive AND a false negative."""
    field_type = SCHEMA[field]
    tp = fp = fn = 0
    for record in records:
        pred, gold = record["pred"][field], record["gold"][field]
        has_pred, has_gold = bool(pred.strip()), bool(gold.strip())
        if has_pred and has_gold and match_value(pred, gold, field_type, mode):
            tp += 1
            continue
        if has_pred:
            fp += 1
        if has_gold:
            fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return FieldScore(field, mode, tp, fp, fn, precision, recall, f1)


def macro_f1(records: Sequence[Mapping], mode: str) -> float:
    """Module 1's unweighted mean of per-field F1 over every field in SCHEMA."""
    if not SCHEMA:
        return 0.0
    return sum(score_field(records, f, mode).f1 for f in SCHEMA) / len(SCHEMA)


def classify_cell(pred: str, gold: str, field_type: str, mode: str) -> str:
    """Module 1's five-way cell label."""
    has_pred = isinstance(pred, str) and bool(pred.strip())
    has_gold = isinstance(gold, str) and bool(gold.strip())
    if not has_pred and not has_gold:
        return "true_negative"
    if not has_pred:
        return "miss"
    if not has_gold:
        return "spurious"
    return "correct" if match_value(pred, gold, field_type, mode) else "wrong_value"


def review_queue(records: Sequence[Mapping], budget: int) -> tuple[tuple[str, str], ...]:
    """Module 1's routing policy: lowest confidence first, ties on (doc_id, field)."""
    if budget < 0:
        raise ValueError(f"budget must be >= 0, got {budget}")
    cells = [(record["conf"][field], record["doc_id"], field)
             for record in records for field in SCHEMA]
    cells.sort(key=lambda cell: (cell[0], cell[1], cell[2]))
    return tuple((doc_id, field) for _, doc_id, field in cells[:budget])


def apply_reviews(records: Sequence[Mapping], routed: Iterable[tuple[str, str]]) -> list[dict]:
    """Module 1's PERFECT reviewer: every routed cell takes the gold value, confidence 1.0."""
    routed_set = set(routed)
    out = []
    for record in records:
        pred, conf = dict(record["pred"]), dict(record["conf"])
        for field in pred:
            if (record["doc_id"], field) in routed_set:
                pred[field] = record["gold"][field]
                conf[field] = 1.0
        out.append({"doc_id": record["doc_id"], "gold": dict(record["gold"]),
                    "pred": pred, "conf": conf})
    return out


# This lesson's review budget: the cells module 1's policy routes to a human in one day.
BUDGET = 240
MACHINE_F1 = macro_f1(RECORDS, "normalised")
PERFECT_REVIEW_F1 = macro_f1(apply_reviews(RECORDS, review_queue(RECORDS, BUDGET)), "normalised")
print(f"the extractor alone             macro F1 {MACHINE_F1:.3f}")
print(f"+ {BUDGET} cells, perfect reviewer   macro F1 {PERFECT_REVIEW_F1:.3f}   <- module 1's promise")

## 2. A reviewer is a rater, not an oracle

A reviewer looking at one routed cell gives one of three **verdicts**:

- `accept` — the machine's value (or its silence) stands;
- `fix` — the reviewer types the value printed on the page;
- `clear` — the field is not on the page, so the machine's value is deleted.

Each cell has a true verdict, read off module 1's labels: a correct cell or a true negative
should be accepted, a miss or a wrong value fixed, a spurious value cleared.

Each simulated reviewer has four properties you can read in the code below. **Accuracy** on
the first item of a shift. **Fatigue**: accuracy lost for every item already reviewed in the
shift, down to a **floor**. Speed, in minutes per item. And, for one of them, a **habit**: a
verdict they never give. When a reviewer is wrong about a cell that needed changing, most of
the time they accept the machine's value — the rubber stamp.

Every number in the next cell is a parameter of this simulation. None of them is a claim
about how accurate or how quickly real people review documents; measure your own team.

In [ ]:
VERDICTS = ("accept", "fix", "clear")
_VERDICT_OF_LABEL = {"correct": "accept", "true_negative": "accept",
                     "miss": "fix", "wrong_value": "fix", "spurious": "clear"}

RUBBER_STAMP_SHARE = 0.70     # a wrong verdict on a cell that needed changing is `accept` this often
FALSE_ALARM_FIX_SHARE = 0.75  # a wrong verdict on a cell that was fine is `fix` this often, else `clear`


def true_verdict(pred: str, gold: str, field_type: str) -> str:
    """The verdict a reviewer who never errs would give, from module 1's cell label."""
    return _VERDICT_OF_LABEL[classify_cell(pred, gold, field_type, "normalised")]


def apply_verdict(pred: str, gold: str, verdict: str) -> str:
    """The cell's value after a verdict: accepted as it is, fixed to the page, or cleared."""
    if verdict not in VERDICTS:
        raise ValueError(f"unknown verdict {verdict!r}; expected one of {VERDICTS}")
    return {"accept": pred, "fix": gold, "clear": ""}[verdict]


class Reviewer(NamedTuple):
    name: str
    tier: int
    accuracy: float           # chance of the right verdict on the first item of a shift
    fatigue: float            # accuracy lost per item already reviewed in the shift
    floor: float              # accuracy never falls below this
    minutes_per_item: float
    never_uses: str           # a verdict this reviewer never gives ("" for none); they accept instead
    seed: int


def accuracy_at(reviewer: Reviewer, position: int) -> float:
    """The reviewer's chance of the right verdict on the item at `position` in the shift (0-based)."""
    return max(reviewer.floor, reviewer.accuracy - reviewer.fatigue * position)


def review_verdict(reviewer: Reviewer, record: Mapping, field: str, position: int) -> str:
    """One reviewer's verdict on one cell, given as the `position`-th item of a shift.

    The dice are fixed per (reviewer, cell): the same reviewer looking at the same cell draws
    the same two numbers whatever the position, so changing a shift length changes how TIRED a
    reviewer is when they meet a cell and nothing else. Two policies compared in this lesson
    differ by their policy, never by luck.
    """
    truth = true_verdict(record["pred"][field], record["gold"][field], SCHEMA[field])
    rng = random.Random(reviewer.seed * 1_000_003 + int(record["doc_id"][4:]) * 10
                        + FIELD_INDEX[field])
    right, which = rng.random(), rng.random()
    if right < accuracy_at(reviewer, position):
        said = truth
    elif truth == "accept":
        said = "fix" if which < FALSE_ALARM_FIX_SHARE else "clear"
    else:
        said = "accept" if which < RUBBER_STAMP_SHARE else ("clear" if truth == "fix" else "fix")
    return "accept" if said == reviewer.never_uses else said


REVIEWER_A = Reviewer("A", 1, accuracy=0.95, fatigue=0.0010, floor=0.70, minutes_per_item=1.5,
                      never_uses="", seed=101)
REVIEWER_B = Reviewer("B", 1, accuracy=0.93, fatigue=0.0008, floor=0.70, minutes_per_item=1.5,
                      never_uses="clear", seed=202)
REVIEWER_S = Reviewer("S", 2, accuracy=0.98, fatigue=0.0002, floor=0.90, minutes_per_item=3.0,
                      never_uses="", seed=303)
TEAM = (REVIEWER_A, REVIEWER_B, REVIEWER_S)

_positions = (0, 40, 80, 120, 160, 240)
print("reviewer  tier  min/item  never says   accuracy at position in shift")
print(" " * 42 + "".join(f"{p:>7d}" for p in _positions))
for _r in TEAM:
    print(f"   {_r.name:6s}{_r.tier:5d}{_r.minutes_per_item:10.1f}  {_r.never_uses or '—':>10s}      "
          + "".join(f"{accuracy_at(_r, p):7.3f}" for p in _positions))

## 3. The audit: two reviewers, every cell

Before a review team goes live, both tier-1 reviewers label the same cells, independently —
here, every cell of the corpus, in document order, in shifts of `AUDIT_SHIFT_ITEMS`. This is
the fixture for sections 4, 5 and 10. Only the simulation knows `AUDIT_TRUTH`; a real team
never sees that column, which is the whole difficulty.

In [ ]:
AUDIT_SHIFT_ITEMS = 120
AUDIT_CELLS = [(record["doc_id"], field) for record in RECORDS for field in SCHEMA]
AUDIT_POSITION = [i % AUDIT_SHIFT_ITEMS for i in range(len(AUDIT_CELLS))]


def run_audit(reviewer: Reviewer) -> list[str]:
    """`reviewer`'s verdict on every cell, in document order, in shifts of AUDIT_SHIFT_ITEMS."""
    return [review_verdict(reviewer, BY_ID[doc_id], field, position)
            for (doc_id, field), position in zip(AUDIT_CELLS, AUDIT_POSITION)]


AUDIT_A = run_audit(REVIEWER_A)
AUDIT_B = run_audit(REVIEWER_B)
AUDIT_TRUTH = [true_verdict(BY_ID[d]["pred"][f], BY_ID[d]["gold"][f], SCHEMA[f])
               for d, f in AUDIT_CELLS]

print(f"{len(AUDIT_CELLS)} cells, each labelled by A and by B\n")
print(f"{'verdict':10s}{'truth':>8s}{'A said':>8s}{'B said':>8s}")
for _v in VERDICTS:
    print(f"{_v:10s}{AUDIT_TRUTH.count(_v):8d}{AUDIT_A.count(_v):8d}{AUDIT_B.count(_v):8d}")
print("\nB never says 'clear'. Any agreement measure you write has to survive a label that one")
print("of the two reviewers never uses.")

## 4. Exercise 1 — `agreement_table` and `raw_agreement`

Two reviewers labelled the same cells in the same order. The first question is how often they
gave the same verdict, and the first instrument is a table of who said what against whom:
one row per label reviewer A used, one column per label reviewer B used — except that a label
only ONE of them used still needs its row and its column, or the table cannot be read across.

<details><summary>💡 Hint 1 — what to think about</summary>

The categories are a property of the PAIR of reviewers, not of either one. And agreement over
nothing, or over two lists of different lengths, is not a number: it is a bug in whoever
called you — a zip that quietly stops at the shorter list hides exactly that bug.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate first: equal lengths, at least one item, otherwise ValueError. The categories are
the sorted union of both lists' distinct labels. Map each category to an index, start from a
square integer array of zeros, and add one at (index of A's label, index of B's label) for
every item. Raw agreement is the number of positions where the two labels are equal, divided
by the number of items — which is also the table's diagonal over its total.

</details>

In [ ]:
def agreement_table(labels_a: Sequence[str], labels_b: Sequence[str]) -> tuple[tuple[str, ...], np.ndarray]:
    """Who said what against whom: counts of (A's label, B's label) over paired items.

    * The categories are the SORTED union of every label either reviewer used — a label only
      one of them used still gets a row and a column, full of zeros where the other never went.
    * Row i is reviewer A saying ``categories[i]``; column j is reviewer B saying
      ``categories[j]``. The array holds integer counts and sums to the number of items.
    * Two lists of different lengths, or two empty lists, are a bug in the caller: raise
      ``ValueError``.

    Returns:
        ``(categories, table)`` — a tuple of labels, and a square ``np.ndarray`` of ints.

    Example:
        >>> cats, table = agreement_table(["accept", "fix", "fix"], ["accept", "fix", "clear"])
        >>> cats
        ('accept', 'clear', 'fix')
        >>> table.tolist()
        [[1, 0, 0], [0, 0, 0], [0, 1, 1]]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def raw_agreement(labels_a: Sequence[str], labels_b: Sequence[str]) -> float:
    """The fraction of paired items on which the two reviewers gave the same label.

    Same validation as `agreement_table`: different lengths or no items raise ``ValueError``.

    Returns:
        A float in [0, 1].

    Example:
        >>> raw_agreement(["accept", "fix", "fix", "clear"], ["accept", "fix", "clear", "clear"])
        0.75
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_agreement() -> None:
    cats, table = agreement_table(["accept", "fix", "fix"], ["accept", "fix", "clear"])
    assert tuple(cats) == ("accept", "clear", "fix"), (
        f"categories came back as {tuple(cats)}; expected ('accept', 'clear', 'fix') — the "
        "SORTED union of both reviewers' labels. 'clear' appears only in B's list and still "
        "needs its own row and column.")
    assert np.asarray(table).tolist() == [[1, 0, 0], [0, 0, 0], [0, 1, 1]], (
        f"table came back as {np.asarray(table).tolist()}; rows are reviewer A, columns "
        "reviewer B, and A's 'fix' against B's 'clear' is row 'fix', column 'clear'")
    assert np.issubdtype(np.asarray(table).dtype, np.integer), (
        f"the table's dtype is {np.asarray(table).dtype}; it holds COUNTS, so integers. "
        "np.zeros gives floats unless you ask for dtype=int.")
    got = raw_agreement(["accept", "fix", "fix", "clear"], ["accept", "fix", "clear", "clear"])
    assert abs(got - 0.75) < 1e-12, (
        f"raw_agreement gave {got!r}; three positions of four match, so 0.75 — a fraction, "
        "not a count")
    for a, b, why in ((["fix"], ["fix", "fix"], "lists of different lengths"),
                      ([], [], "two empty lists")):
        for fn in (agreement_table, raw_agreement):
            try:
                fn(a, b)
            except ValueError:
                continue
            raise AssertionError(f"{fn.__name__} accepted {why}; raise ValueError — a zip "
                                 "that stops at the shorter list hides a misaligned join")
    print("exercise 1 looks right")


_try("exercise 1", _check_agreement)

Here is the audit through your table. Then the same instrument pointed at fatigue: agreement
between A and B at each stretch of the shift — the only column a real team can compute —
beside each reviewer's accuracy against the truth, which only this simulation can.

In [ ]:
def _show_audit_table() -> None:
    cats, table = agreement_table(AUDIT_A, AUDIT_B)
    print("rows: what A said · columns: what B said")
    print(f"{'':10s}" + "".join(f"{c:>9s}" for c in cats))
    for label, row in zip(cats, table):
        print(f"{label:10s}" + "".join(f"{int(v):9d}" for v in row))
    print(f"\nraw agreement {raw_agreement(AUDIT_A, AUDIT_B):.3f} over {len(AUDIT_A)} cells")

    width = 20
    print(f"\n{'position in shift':>18s}{'cells':>7s}{'A agrees with B':>17s}"
          f"{'A vs truth':>12s}{'B vs truth':>12s}")
    for start in range(0, AUDIT_SHIFT_ITEMS, width):
        idx = [i for i, p in enumerate(AUDIT_POSITION) if start <= p < start + width]
        a = [AUDIT_A[i] for i in idx]
        b = [AUDIT_B[i] for i in idx]
        t = [AUDIT_TRUTH[i] for i in idx]
        print(f"{start:>8d} to {start + width - 1:<6d}{len(idx):7d}{raw_agreement(a, b):17.3f}"
              f"{raw_agreement(t, a):12.3f}{raw_agreement(t, b):12.3f}")
    early = [i for i, p in enumerate(AUDIT_POSITION) if p < AUDIT_SHIFT_ITEMS // 2]
    late = [i for i, p in enumerate(AUDIT_POSITION) if p >= AUDIT_SHIFT_ITEMS // 2]
    for name, idx in (("first half of each shift", early), ("second half of each shift", late)):
        a, b, t = ([lst[i] for i in idx] for lst in (AUDIT_A, AUDIT_B, AUDIT_TRUTH))
        print(f"{name:27s} A-B agreement {raw_agreement(a, b):.3f}   A vs truth {raw_agreement(t, a):.3f}")
    print("\nYou cannot see the truth, but you can see disagreement, and it moves with accuracy:")
    print("a disagreement rate that climbs late in a shift is fatigue you can measure.")


_try("audit table", _show_audit_table, needs=("exercise 1",))

## 5. Exercise 2 — `chance_agreement` and `cohens_kappa`

Raw agreement flatters a skewed label mix. If most cells should be accepted, two reviewers who
accept almost everything agree almost always, whether or not they looked. Cohen's kappa
corrects for that: κ = (p_o − p_e) / (1 − p_e), where p_o is the observed agreement and p_e
the agreement expected if each reviewer labelled at random with their OWN label frequencies
(scikit-learn's reference documentation, citing Cohen 1960; see `claims.yaml`). κ = 1 is
complete agreement, 0 is no better than chance, and below 0 is worse than chance.

<details><summary>💡 Hint 1 — what to think about</summary>

p_e is a sum over labels of (how often A used it) times (how often B used it), matched BY
LABEL, over every label either of them used. Two per-reviewer frequency lists sorted
separately line up only when both reviewers used exactly the same labels — the audit shows
they do not. Then think about the one case the formula cannot answer: when is 1 − p_e zero,
and why would reporting perfect agreement, or none, both be dishonest there?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate exactly as in exercise 1. For each label in the union, multiply A's share of items
with that label by B's share, and add the products; that is `chance_agreement`. For
`cohens_kappa`, take raw agreement and chance agreement; if chance agreement is 1 (both
reviewers gave one and the same label to everything) return the value the docstring names
for undefined; otherwise apply the formula as it stands — no clamping, no absolute value.

</details>

In [ ]:
def chance_agreement(labels_a: Sequence[str], labels_b: Sequence[str]) -> float:
    """p_e: the agreement two reviewers would reach by labelling at random, each with their own
    label frequencies.

    The sum, over every label EITHER reviewer used, of (A's share of items with that label)
    times (B's share of items with that label). A label only one reviewer used contributes
    zero, and must not shift any other label's pairing. Different lengths or no items raise
    ``ValueError``.

    Returns:
        A float in [0, 1].

    Example:
        >>> chance_agreement(["accept", "accept", "fix", "fix"], ["accept", "fix", "fix", "fix"])
        0.5
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cohens_kappa(labels_a: Sequence[str], labels_b: Sequence[str]) -> float:
    """Cohen's kappa between two reviewers: (p_o − p_e) / (1 − p_e).

    p_o is `raw_agreement`, p_e is `chance_agreement`.

    * Identical labels, with at least two different labels in use, give exactly 1.0.
    * Agreement no better than chance gives 0.0; systematic disagreement gives a NEGATIVE
      kappa. Do not clamp and do not take an absolute value.
    * When p_e is 1 — both reviewers gave one and the same label to every item — kappa is 0/0.
      Return ``float("nan")``: agreeing on the only label anybody used says nothing about
      whether either reviewer can tell labels apart, so 1.0 would claim a reliability nobody
      measured.
    * Different lengths or no items raise ``ValueError``.

    Returns:
        A float in [-1, 1], or nan.

    Example:
        >>> cohens_kappa(["accept", "accept", "fix", "fix"], ["accept", "fix", "accept", "fix"])
        0.0
        >>> cohens_kappa(["fix", "accept"], ["accept", "fix"])
        -1.0
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_kappa() -> None:
    same = ["accept", "fix", "clear", "accept"]
    got = cohens_kappa(same, same)
    assert abs(got - 1.0) < 1e-12, f"identical labels must give kappa 1.0, got {got!r}"
    got = cohens_kappa(["accept", "accept", "fix", "fix"], ["accept", "fix", "accept", "fix"])
    assert abs(got) < 1e-12, (
        f"got {got!r}. Observed and chance agreement are both 0.5 here, so kappa is 0.0 — if "
        "you returned 0.5 you returned raw agreement, not kappa")
    got = cohens_kappa(["fix", "accept", "fix", "accept"], ["accept", "fix", "accept", "fix"])
    assert abs(got + 1.0) < 1e-12, (
        f"got {got!r}. These two reviewers disagree on every item, which is worse than chance: "
        "kappa is -1.0. Do not clamp at zero and do not take an absolute value.")
    a = ["accept", "accept", "fix", "fix", "accept", "fix"]
    b = ["accept", "clear", "clear", "clear", "accept", "accept"]   # B never says 'fix'
    pe = chance_agreement(a, b)
    assert abs(pe - 0.25) < 1e-12, (
        f"chance_agreement gave {pe!r}; expected 0.25. Only 'accept' is used by both, so only "
        "'accept' contributes: (3/6) x (3/6). 0.5 means the two reviewers' frequencies were "
        "paired by POSITION in their own sorted lists — A's 'fix' against B's 'clear' — "
        "instead of by label.")
    got = cohens_kappa(a, b)
    assert abs(got - 1 / 9) < 1e-12, f"kappa for that pair is 1/9, got {got!r}"
    got = cohens_kappa(["accept"] * 5, ["accept"] * 5)
    assert isinstance(got, float) and math.isnan(got), (
        f"got {got!r}. Both reviewers used one label for everything, so p_e = 1 and kappa is "
        "0/0: return float('nan'), not 1.0 and not 0.0")
    print("exercise 2 looks right")


_try("exercise 2", _check_kappa)

The audit, chance-corrected — and, beside it, what a reviewer who never looked would score.

In [ ]:
def _show_kappa_on_audit() -> None:
    p_o, p_e = raw_agreement(AUDIT_A, AUDIT_B), chance_agreement(AUDIT_A, AUDIT_B)
    kappa = cohens_kappa(AUDIT_A, AUDIT_B)
    print(f"A against B   raw agreement {p_o:.3f}   chance agreement {p_e:.3f}   kappa {kappa:.3f}")
    lazy = ["accept"] * len(AUDIT_TRUTH)
    print(f"'accept everything' against the truth   raw {raw_agreement(AUDIT_TRUTH, lazy):.3f}   "
          f"kappa {cohens_kappa(AUDIT_TRUTH, lazy):.3f}")
    print(f"A against the truth (simulation only)  raw {raw_agreement(AUDIT_TRUTH, AUDIT_A):.3f}   "
          f"kappa {cohens_kappa(AUDIT_TRUTH, AUDIT_A):.3f}")
    print("\nA reviewer who never opened a document would 'agree' with the truth on every cell that")
    print("should be accepted. Raw agreement pays them for it; kappa pays them nothing.")


_try("kappa on the audit", _show_kappa_on_audit, needs=("exercise 1", "exercise 2"))

## 6. The day's queue, with deadlines

Now the reviewers go to work. The day's queue is module 1's: the `BUDGET` lowest-confidence
cells, ranked by `review_queue`. Documents arrive steadily over `INTAKE_MINUTES` minutes, and
each routed cell arrives with its document. Every cell carries a **deadline**: a money
cell must be reviewed within `SLA_MINUTES["money"]` minutes of arriving, because a payment run
is waiting on it; everything else within `SLA_MINUTES["id"]`. Finishing after the deadline is a
**breach**. The SLA figures are this lesson's parameters, not anybody's service levels.

In [ ]:
INTAKE_MINUTES = 240          # the documents arrive evenly over this many minutes
SLA_MINUTES = {"money": 45, "date": 180, "id": 180, "integer": 180, "text": 180}
BREAK_MINUTES = 15            # a reviewer's break between shifts
DEFAULT_SHIFT_ITEMS = 60      # items per tier-1 shift until section 9 measures the choice
TIER2_SHIFT_ITEMS = 60


class QueueItem(NamedTuple):
    doc_id: str
    field: str
    arrival: float            # minute the cell enters the queue
    deadline: float           # minute by which its review must be finished
    conf: float               # the extractor's confidence, as in module 1


class Served(NamedTuple):
    doc_id: str
    field: str
    arrival: float
    deadline: float
    start: float              # minute the reviewer started the item
    finish: float             # minute the reviewer finished it
    position: int             # items this reviewer had done since their last break (0-based)


class BreachReport(NamedTuple):
    n_items: int
    n_breached: int
    breach_rate: float
    worst_lateness: float     # minutes past the deadline of the latest item, 0.0 if none late


def build_day(records: Sequence[Mapping], budget: int) -> list[QueueItem]:
    """Module 1's routed cells as a day's queue: arrival with their document, deadline by type."""
    order = {record["doc_id"]: i for i, record in enumerate(records)}
    by_id = {record["doc_id"]: record for record in records}
    items = []
    for doc_id, field in review_queue(records, budget):
        arrival = order[doc_id] * INTAKE_MINUTES // len(records)
        items.append(QueueItem(doc_id, field, arrival,
                               arrival + SLA_MINUTES[SCHEMA[field]], by_id[doc_id]["conf"][field]))
    return items


DAY = build_day(RECORDS, BUDGET)
_money = sum(SCHEMA[it.field] == "money" for it in DAY)
print(f"{len(DAY)} cells routed, {_money} of them money; arrivals from minute "
      f"{min(it.arrival for it in DAY):.0f} to {max(it.arrival for it in DAY):.0f}")
print(f"tier-1 work at {REVIEWER_A.minutes_per_item} min per cell: "
      f"{len(DAY) * REVIEWER_A.minutes_per_item:.0f} minutes, before a single break")
print("true verdicts in the queue: " + ", ".join(
    f"{v} {sum(true_verdict(BY_ID[it.doc_id]['pred'][it.field], BY_ID[it.doc_id]['gold'][it.field], SCHEMA[it.field]) == v for it in DAY)}"
    for v in VERDICTS))

## 7. Exercise 3 — `sla_schedule` and `breach_report`

One reviewer works the queue. Whenever they are free, they take the most urgent cell that has
**already arrived** — earliest deadline first — and nothing that has not. Ties break on lower
confidence, then `(doc_id, field)`, module 1's order. After every `shift_items` items they
take a break of `break_minutes`, and their count of items since the last break — the
`position` that drives fatigue — starts again at zero. Waiting for work to arrive is not a
break.

<details><summary>💡 Hint 1 — what to think about</summary>

This is a clock and a loop, not a sort. Sorting every item by deadline once and serving that
list in order serves cells before they exist. Ask at each step: what time is it, which items
are in the building, and what happens when none is? Watch the break's off-by-one — it comes
after the `shift_items`-th item, not before the first and not after one more. "Late" means
finished strictly after the deadline. And when you jump the clock to the next arrival, that
item must then count as arrived — test `<` where the docstring says `<=` and the loop never
ends.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the three parameters. Keep a clock and a position counter, both starting where the
docstring says, and the items not yet served (a new list — never pop from the caller's).
Loop while any remain: the ready ones are those whose arrival is at or before the clock; if
none is ready, move the clock to the earliest remaining arrival and loop again. Otherwise
take the ready item with the smallest (deadline, conf, doc_id, field), record it with
start = clock, finish = clock + minutes, and the current position; advance the clock and the
position; when the position reaches `shift_items`, add the break to the clock and start the
count again. For `breach_report`, count the items that finished after their deadline and
keep the largest overrun.

</details>

In [ ]:
def sla_schedule(items: Sequence[QueueItem], minutes_per_item: float, shift_items: int,
                 break_minutes: float) -> list[Served]:
    """One reviewer working `items`, earliest deadline first, in shifts. Returns what was served, in order.

    * The reviewer is at the desk from minute 0. Whenever free, they serve the item with the
      smallest ``(deadline, conf, doc_id, field)`` among the items that have ARRIVED
      (``arrival <= clock``). If none has arrived, they wait until the next arrival.
    * Each item takes `minutes_per_item`: ``finish = start + minutes_per_item``.
    * ``position`` is the number of items served since the last break, starting at 0. After the
      `shift_items`-th item of a shift, the next item cannot start before that item's finish
      plus `break_minutes`, and the position starts again at 0. Waiting for arrivals is not a
      break and does not reset the position.
    * `items` is not modified. No items gives ``[]``.
    * ``minutes_per_item <= 0``, ``shift_items < 1`` or ``break_minutes < 0`` raises ``ValueError``.

    Returns:
        A list of ``Served``, one per item, in the order the reviewer served them.

    Example:
        >>> day = [QueueItem("D1", "currency", 0, 100, 0.5),
        ...        QueueItem("D2", "total_amount", 0, 30, 0.9),
        ...        QueueItem("D3", "invoice_id", 5, 10, 0.1)]
        >>> [(s.doc_id, s.start, s.finish) for s in sla_schedule(day, 2.0, 50, 0.0)]
        [('D2', 0.0, 2.0), ('D1', 2.0, 4.0), ('D3', 5.0, 7.0)]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def breach_report(served: Sequence[Served]) -> BreachReport:
    """How many served items finished after their deadline, and by how much at worst.

    An item is breached when ``finish > deadline`` — finishing exactly at the deadline is on
    time. ``breach_rate`` is breached over served, and 0.0 when nothing was served.
    ``worst_lateness`` is the largest ``finish - deadline`` among breached items, 0.0 if none.

    Returns:
        A ``BreachReport(n_items, n_breached, breach_rate, worst_lateness)``.

    Example:
        >>> breach_report([Served("D1", "currency", 0, 10, 8, 10, 0),
        ...                Served("D2", "currency", 0, 10, 10, 12, 1)])
        BreachReport(n_items=2, n_breached=1, breach_rate=0.5, worst_lateness=2)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_schedule() -> None:
    day = [QueueItem("D1", "currency", 0, 100, 0.5),
           QueueItem("D2", "total_amount", 0, 30, 0.9),
           QueueItem("D3", "invoice_id", 5, 10, 0.1)]
    snapshot = list(day)
    served = sla_schedule(day, 2.0, 50, 0.0)
    got = [(s.doc_id, s.start, s.finish) for s in served]
    assert [g[0] for g in got] == ["D2", "D1", "D3"], (
        f"served in the order {[g[0] for g in got]}; expected D2, D1, D3. At minute 0 D2 has the "
        "earliest deadline of the two that have arrived. At minute 2, D3 is the most urgent "
        "item but it does not arrive until minute 5 — serve D1, which is in the building.")
    assert got[2][1:] == (5.0, 7.0), (
        f"D3 started at {got[2][1]}; nothing is ready at minute 4, so the reviewer waits for "
        "D3's arrival at minute 5 and starts it then")
    assert day == snapshot, "sla_schedule must not modify the list it was given"
    tied = [QueueItem("D9", "currency", 0, 50, 0.4), QueueItem("D1", "currency", 0, 50, 0.4),
            QueueItem("D5", "currency", 0, 50, 0.2)]
    order = [s.doc_id for s in sla_schedule(tied, 1.0, 50, 0.0)]
    assert order == ["D5", "D1", "D9"], (
        f"equal deadlines served as {order}; ties break on lower confidence first, then "
        "(doc_id, field)")
    shift = [QueueItem(f"D{i}", "currency", 0, 100, 0.5) for i in range(3)]
    s3 = sla_schedule(shift, 1.0, 2, 5.0)
    assert [(s.start, s.position) for s in s3] == [(0.0, 0), (1.0, 1), (7.0, 0)], (
        f"got (start, position) {[(s.start, s.position) for s in s3]}; with shifts of 2 items "
        "and a 5-minute break, the third item starts at 1 + 1 + 5 = 7 and is position 0 of the "
        "next shift")
    report = breach_report([Served("D1", "currency", 0, 10, 8, 10, 0),
                            Served("D2", "currency", 0, 10, 10, 12, 1)])
    assert (report.n_items, report.n_breached) == (2, 1), (
        f"got {report}; finishing exactly AT the deadline is on time — only finish > deadline "
        "is a breach")
    assert abs(report.breach_rate - 0.5) < 1e-12 and abs(report.worst_lateness - 2) < 1e-12, \
        f"one of two late, by 2 minutes: got {report}"
    assert breach_report([]).breach_rate == 0.0, "nothing served, nothing breached: rate 0.0"
    print("exercise 3 looks right")


_try("exercise 3", _check_schedule)

Your scheduler serves earliest deadline first. Here is what that choice is worth on the day's
queue, against first-in-first-out and against module 1's lowest-confidence-first order — the
same reviewer, the same shifts, only the order changes.

In [ ]:
def served_in_order(items: Sequence[QueueItem], sort_key: Callable[[QueueItem], float] | None,
                    shift_items: int, reviewer: Reviewer = REVIEWER_A) -> list[Served]:
    """Your `sla_schedule`, serving in the order `sort_key` gives instead of by deadline.

    The trick: hand the scheduler a copy of each item whose `deadline` slot holds the sort key,
    then put the true deadline back on what it served, so breaches are counted honestly.
    """
    if sort_key is None:
        return sla_schedule(items, reviewer.minutes_per_item, shift_items, BREAK_MINUTES)
    true_deadline = {(it.doc_id, it.field): it.deadline for it in items}
    proxy = [it._replace(deadline=sort_key(it)) for it in items]
    served = sla_schedule(proxy, reviewer.minutes_per_item, shift_items, BREAK_MINUTES)
    return [s._replace(deadline=true_deadline[(s.doc_id, s.field)]) for s in served]


def _show_orders() -> None:
    print(f"{'order':30s}{'breached':>10s}{'rate':>8s}{'worst, min late':>17s}{'money late':>12s}")
    for name, key in (("earliest deadline first", None),
                      ("first in, first out", lambda it: it.arrival),
                      ("lowest confidence first", lambda it: it.conf)):
        served = served_in_order(DAY, key, DEFAULT_SHIFT_ITEMS)
        report = breach_report(served)
        money_late = sum(s.finish > s.deadline for s in served if SCHEMA[s.field] == "money")
        print(f"{name:30s}{report.n_breached:10d}{report.breach_rate:8.3f}"
              f"{report.worst_lateness:17.1f}{money_late:12d}")
    print("\nModule 1's order is the right one for deciding WHAT to review. It is the wrong one for")
    print("deciding WHEN: it knows nothing about the payment run waiting on a money cell.")


_try("queue orders", _show_orders, needs=("exercise 3",))

## 8. Exercise 4 — `escalate`, the two-tier policy

Tier 1 reviews everything in the queue. Tier 2 — reviewer S, slower and more accurate —
re-reviews what the policy escalates, and S's verdict replaces tier 1's. Senior time is
scarce, so the policy has to send the verdicts most likely to be wrong, or most costly if they
are. Two rules:

1. **Four eyes on money.** Any verdict that changes a money value — `fix` or `clear` on a field
   whose type is in `FOUR_EYES_TYPES` — goes to tier 2.
2. **The rubber-stamp guard.** An `accept` of a cell the machine itself was unsure of —
   confidence strictly below `RUBBER_STAMP_CONF` — goes to tier 2. Article 14(4)(b) of the EU
   AI Act names this failure, over-reliance on a system's output, as "automation bias", and
   asks that the people overseeing a high-risk system be enabled to remain aware of it (see
   `claims.yaml`). A tired reviewer accepting a doubtful value is the shape it takes here.

Everything else stays with tier 1.

<details><summary>💡 Hint 1 — what to think about</summary>

Which verdicts change a value? There are two, not one. And what should happen to a verdict
the policy has never heard of — say "Fix" with a capital F, from a tool that capitalises?
Quietly answering False sends a possibly wrong money change straight to the payment run.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Reject a verdict outside VERDICTS and a field type outside SCHEMA's values with ValueError.
Then return True when the verdict is anything other than accept and the field type is in
FOUR_EYES_TYPES; True when the verdict is accept and the confidence is below
RUBBER_STAMP_CONF; and False otherwise. Name the two constants, never their current values:
whoever retunes the policy changes the constants, not your function.

</details>

In [ ]:
FOUR_EYES_TYPES = frozenset({"money"})
RUBBER_STAMP_CONF = 0.5


def escalate(verdict: str, field_type: str, conf: float) -> bool:
    """Should tier 2 re-review this tier-1 verdict?

    * ``True`` when the verdict changes a value of a type in `FOUR_EYES_TYPES` — ``"fix"`` or
      ``"clear"`` on a money field.
    * ``True`` when the verdict is ``"accept"`` and the extractor's confidence `conf` is
      strictly below `RUBBER_STAMP_CONF` — the machine doubted the value, the human waved it
      through.
    * ``False`` otherwise.
    * A `verdict` not in `VERDICTS`, or a `field_type` not among SCHEMA's types, raises
      ``ValueError``: a policy must not answer for a label it does not know.

    Returns:
        A bool.

    Example:
        >>> escalate("fix", "money", 0.95), escalate("accept", "text", 0.1)
        (True, True)
        >>> escalate("fix", "text", 0.1), escalate("accept", "money", 0.9)
        (False, False)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_escalate() -> None:
    assert escalate("fix", "money", 0.99) is True, "a fix on a money field goes to tier 2"
    assert escalate("clear", "money", 0.99) is True, (
        "a clear on a money field deletes an amount — that changes money too, so it goes to "
        "tier 2; checking only for 'fix' misses it")
    assert escalate("accept", "money", 0.9) is False, \
        "accepting a confident money value changes nothing and needs no second reviewer"
    assert escalate("accept", "text", 0.0) is True, (
        "accepting a cell the machine had zero confidence in is the rubber stamp — escalate, "
        "whatever the field type")
    assert escalate("accept", "id", RUBBER_STAMP_CONF) is False, (
        "confidence exactly AT RUBBER_STAMP_CONF is not below it: compare with <, not <=")
    assert escalate("fix", "text", 0.1) is False, (
        "a fix on a non-money field stays with tier 1: the reviewer and the machine agree the "
        "value was doubtful")
    for verdict, field_type in (("Fix", "money"), ("approve", "text"), ("accept", "phone")):
        try:
            escalate(verdict, field_type, 0.5)
        except ValueError:
            continue
        raise AssertionError(f"escalate({verdict!r}, {field_type!r}, ...) must raise ValueError "
                             "— a label the policy does not know is not a 'no'")
    print("exercise 4 looks right")


_try("exercise 4", _check_escalate)

The whole day, through both tiers: tier 1 works the queue with your scheduler, your policy
picks what goes up, and tier 2 works that with the same scheduler — an escalated cell arrives
at tier 2 when tier 1 finishes it, with its original deadline. Beside it: no escalation, and a
control that escalates the same NUMBER of cells at random.

One day is one throw of the reviewers' dice, and a day's queue holds only a few dozen
mistakes, so every row is averaged over `N_DAYS` simulated days: the same queue and the same
reviewers, with fresh dice each day.

In [ ]:
N_DAYS = 20


class DayResult(NamedTuple):
    shift_items: int
    per_hour: float           # tier-1 items per hour at the desk, breaks included
    tier1_wrong: int          # tier-1 verdicts that did not match the truth
    escalated: int
    caught: int               # wrong tier-1 verdicts that tier 2 saw
    still_wrong: int          # routed cells whose final value is still wrong
    breach: BreachReport      # end to end: the last tier to touch a cell decides
    macro_f1: float           # module 1's harness, against the truth


class DaySummary(NamedTuple):
    """Means over N_DAYS days of one rota and one policy; breach_worst is the worst single day."""
    shift_items: int
    per_hour: float
    tier1_accuracy: float
    tier1_wrong: float
    escalated: float
    caught: float
    still_wrong: float
    breach_rate: float
    breach_worst: float
    macro_f1: float


def _on_day(reviewer: Reviewer, day: int) -> Reviewer:
    """The same reviewer on another day: same skill, same fatigue, fresh dice."""
    return reviewer._replace(seed=reviewer.seed + 7919 * day)


def run_day(shift_items: int, policy: str = "two-tier", day: int = 0) -> DayResult:
    """One day of the queue. `policy` is "two-tier", "tier 1 only" or "random" (same count)."""
    tier1, tier2 = _on_day(REVIEWER_A, day), _on_day(REVIEWER_S, day)
    first = sla_schedule(DAY, tier1.minutes_per_item, shift_items, BREAK_MINUTES)
    verdict, truth, last = {}, {}, {}
    for s in first:
        key, record = (s.doc_id, s.field), BY_ID[s.doc_id]
        verdict[key] = review_verdict(tier1, record, s.field, s.position)
        truth[key] = true_verdict(record["pred"][s.field], record["gold"][s.field], SCHEMA[s.field])
        last[key] = s
    tier1_wrong = sum(verdict[k] != truth[k] for k in verdict)
    rule = [s for s in first
            if escalate(verdict[(s.doc_id, s.field)], SCHEMA[s.field], BY_ID[s.doc_id]["conf"][s.field])]
    if policy == "two-tier":
        up = rule
    elif policy == "random":
        up = sorted(random.Random(day).sample(first, len(rule)), key=lambda s: s.finish)
    else:
        up = []
    second = sla_schedule([QueueItem(s.doc_id, s.field, s.finish, s.deadline,
                                     BY_ID[s.doc_id]["conf"][s.field]) for s in up],
                          tier2.minutes_per_item, TIER2_SHIFT_ITEMS, BREAK_MINUTES)
    caught = 0
    for s in second:
        key = (s.doc_id, s.field)
        caught += verdict[key] != truth[key]
        verdict[key] = review_verdict(tier2, BY_ID[s.doc_id], s.field, s.position)
        last[key] = s
    after, still_wrong = [], 0
    for record in RECORDS:
        pred = dict(record["pred"])
        for field in SCHEMA:
            key = (record["doc_id"], field)
            if key in verdict:
                pred[field] = apply_verdict(record["pred"][field], record["gold"][field], verdict[key])
                still_wrong += classify_cell(pred[field], record["gold"][field], SCHEMA[field],
                                             "normalised") in ("miss", "spurious", "wrong_value")
        after.append({"doc_id": record["doc_id"], "gold": dict(record["gold"]), "pred": pred,
                      "conf": dict(record["conf"])})
    breaks = sum(1 for s in first[1:] if s.position == 0)
    desk_minutes = len(first) * tier1.minutes_per_item + breaks * BREAK_MINUTES
    return DayResult(shift_items, 60 * len(first) / desk_minutes, tier1_wrong, len(up), caught,
                     still_wrong, breach_report(list(last.values())), macro_f1(after, "normalised"))


def summarise(shift_items: int, policy: str = "two-tier", days: int = N_DAYS) -> DaySummary:
    """`run_day` on `days` days, averaged."""
    runs = [run_day(shift_items, policy, day) for day in range(days)]

    def mean(values) -> float:
        return float(np.mean(list(values)))

    return DaySummary(shift_items, mean(r.per_hour for r in runs),
                      1 - mean(r.tier1_wrong for r in runs) / len(DAY),
                      mean(r.tier1_wrong for r in runs), mean(r.escalated for r in runs),
                      mean(r.caught for r in runs), mean(r.still_wrong for r in runs),
                      mean(r.breach.breach_rate for r in runs),
                      max(r.breach.breach_rate for r in runs), mean(r.macro_f1 for r in runs))


def _show_two_tier() -> None:
    wrong_before = sum(true_verdict(BY_ID[it.doc_id]["pred"][it.field], BY_ID[it.doc_id]["gold"][it.field],
                                    SCHEMA[it.field]) != "accept" for it in DAY)
    print(f"shifts of {DEFAULT_SHIFT_ITEMS} items · {wrong_before} of the {len(DAY)} routed cells are "
          f"wrong before anyone looks · means over {N_DAYS} days\n")
    print(f"{'policy':14s}{'escalated':>10s}{'tier-1 mistakes seen':>22s}{'still wrong':>13s}"
          f"{'breach rate':>13s}{'macro F1':>10s}")
    rows = {}
    for policy in ("tier 1 only", "two-tier", "random"):
        r = rows[policy] = summarise(DEFAULT_SHIFT_ITEMS, policy)
        print(f"{policy:14s}{r.escalated:10.1f}{f'{r.caught:.1f} of {r.tier1_wrong:.1f}':>22s}"
              f"{r.still_wrong:13.1f}{r.breach_rate:13.3f}{r.macro_f1:10.3f}")
    print(f"\nmodule 1's perfect reviewer, same budget: macro F1 {PERFECT_REVIEW_F1:.3f}")
    rules, rand = rows["two-tier"], rows["random"]
    print(f"The same number of senior reviews finds {rules.caught:.1f} tier-1 mistakes a day when the "
          f"rules choose them\nand {rand.caught:.1f} when chance does. A policy has to beat random at "
          "the same cost before it\nis worth its complexity.")


_try("two tiers", _show_two_tier, needs=("exercise 3", "exercise 4"))

## 9. Throughput against quality

The one lever this lesson has not pulled is the shift. Long shifts waste no time on breaks, so
the queue drains fast and deadlines hold — but a reviewer deep into a long shift is the tired
one from section 4. Short shifts keep reviewers fresh and spend the day on breaks. Below, the
whole two-tier day is re-run for each shift length in `SHIFT_LENGTHS`, and the policy picks
the best quality among the shift lengths whose breach rate meets `BREACH_TARGET`. The target
is this lesson's; yours comes from whoever is waiting on the queue.

In [ ]:
SHIFT_LENGTHS = (10, 20, 40, 60, 80, 120, 160, 240)
BREACH_TARGET = 0.05


def sweep_shifts() -> list[DaySummary]:
    """The two-tier day at every shift length in SHIFT_LENGTHS, each averaged over N_DAYS days."""
    return [summarise(n) for n in SHIFT_LENGTHS]


def choose_shift(results: Sequence[DaySummary]) -> DaySummary | None:
    """Fewest cells still wrong among the rotas whose mean breach rate meets BREACH_TARGET.

    Ties go to the higher macro F1, then to the longer shift (more items per hour).
    """
    ok = [r for r in results if r.breach_rate <= BREACH_TARGET]
    return min(ok, key=lambda r: (r.still_wrong, -r.macro_f1, -r.shift_items)) if ok else None


def _show_sweep() -> None:
    results = sweep_shifts()
    print(f"two-tier policy, means over {N_DAYS} days\n")
    print(f"{'shift':>6s}{'items per hour':>16s}{'tier-1 right':>14s}{'escalated':>11s}"
          f"{'still wrong':>13s}{'breach rate':>13s}{'macro F1':>10s}")
    for r in results:
        print(f"{r.shift_items:6d}{r.per_hour:16.1f}{r.tier1_accuracy:14.3f}{r.escalated:11.1f}"
              f"{r.still_wrong:13.1f}{r.breach_rate:13.3f}{r.macro_f1:10.3f}")
    chosen = choose_shift(results)
    if chosen is None:
        print(f"\nno shift length meets a breach rate of {BREACH_TARGET}: this queue needs more "
              "reviewers, not a better rota")
        return
    fastest = max(results, key=lambda r: r.per_hour)
    freshest = min(results, key=lambda r: r.still_wrong)
    print(f"\nchosen: shifts of {chosen.shift_items} items — {chosen.per_hour:.1f} items per hour, "
          f"breach rate {chosen.breach_rate:.3f}, {chosen.still_wrong:.1f} cells still wrong")
    print(f"fastest rota:  {fastest.shift_items} items, {fastest.per_hour:.1f} per hour, "
          f"{fastest.still_wrong:.1f} cells still wrong")
    print(f"freshest rota: {freshest.shift_items} items, {freshest.still_wrong:.1f} cells still wrong, "
          f"breach rate {freshest.breach_rate:.3f}")
    spread = max(r.macro_f1 for r in results) - min(r.macro_f1 for r in results)
    print(f"across the rotas, cells still wrong range from {freshest.still_wrong:.1f} to "
          f"{max(r.still_wrong for r in results):.1f} a day while corpus macro F1 moves by "
          f"{spread:.3f}:\nthe routed cells are a small slice of the corpus, so judge a rota on the "
          "cells it touched.")
    print("Throughput and quality are bought with the same minutes. The table prices each one in")
    print("the other's currency; the target decides which row you can afford.")


_try("throughput against quality", _show_sweep, needs=("exercise 3", "exercise 4"))

## 10. Exercise 5 — `harness_ceiling`, the uncomfortable measurement

Every score your harness reports is agreement with gold, and gold is written by reviewers
like A. Suppose their mistakes are independent and A and B are alike. Then each agrees with
the truth, beyond chance, by some factor c — and for A and B to agree with EACH OTHER beyond
chance, both have to be tracking the truth, so their kappa is about c × c. You measured that
kappa. So c ≈ √κ: the kappa that a **perfect** system, one that always knows the truth, can
reach against gold one of these reviewers wrote. Put back on the raw-agreement scale with the
same chance agreement, it is p_e + √κ · (1 − p_e).

That is the ceiling your harness can see. Not 1.0. A system scoring above it is not better
than perfect; it is copying your annotators' mistakes.

<details><summary>💡 Hint 1 — what to think about</summary>

Three numbers are easy to confuse here: the reviewers' kappa, the reviewers' raw agreement,
and the ceiling. The first two describe what a system as good as a SECOND reviewer would
score. The ceiling describes a perfect one, so it sits above both. Then the edges: a kappa at
or below zero leaves no signal to take a root of, and a kappa that is nan leaves no ceiling.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Get kappa from your `cohens_kappa`; if it is nan, raise ValueError. The kappa ceiling is the
square root of kappa when kappa is positive, and the docstring's floor otherwise. Get the
chance agreement from your `chance_agreement`; the accuracy ceiling is that chance agreement
plus the kappa ceiling times one minus the chance agreement. Return all three in a `Ceiling`.

</details>

In [ ]:
class Ceiling(NamedTuple):
    kappa: float              # the two reviewers' kappa
    kappa_ceiling: float      # the kappa a perfect system can reach against one reviewer's gold
    accuracy_ceiling: float   # the same ceiling on the raw-agreement scale


def harness_ceiling(labels_a: Sequence[str], labels_b: Sequence[str]) -> Ceiling:
    """The best score a perfect system can reach against gold written by one of two reviewers.

    From the two reviewers' labels on the same items:

    * ``kappa`` is `cohens_kappa` of the two.
    * ``kappa_ceiling`` is ``sqrt(kappa)`` when kappa is positive, and 0.0 when it is zero or
      negative — reviewers who agree no better than chance have written gold that carries no
      signal about the truth.
    * ``accuracy_ceiling`` is ``p_e + kappa_ceiling * (1 - p_e)``, with ``p_e`` the pair's
      `chance_agreement`: the ceiling put back on the raw-agreement scale.
    * A nan kappa (one label used for everything) raises ``ValueError``: there is no ceiling to
      report. Different lengths or no items raise ``ValueError`` as before.

    Returns:
        A ``Ceiling(kappa, kappa_ceiling, accuracy_ceiling)``.

    Example:
        >>> a = ["fix"] * 41 + ["accept"] * 41 + ["fix"] * 9 + ["accept"] * 9
        >>> b = ["fix"] * 41 + ["accept"] * 41 + ["accept"] * 9 + ["fix"] * 9
        >>> c = harness_ceiling(a, b)          # kappa 0.64, chance agreement 0.5
        >>> round(c.kappa_ceiling, 6), round(c.accuracy_ceiling, 6)
        (0.8, 0.9)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_ceiling() -> None:
    a = ["fix"] * 41 + ["accept"] * 41 + ["fix"] * 9 + ["accept"] * 9
    b = ["fix"] * 41 + ["accept"] * 41 + ["accept"] * 9 + ["fix"] * 9
    c = harness_ceiling(a, b)
    assert abs(c.kappa - 0.64) < 1e-9, f"kappa for this pair is 0.64, got {c.kappa!r}"
    assert abs(c.kappa_ceiling - 0.8) < 1e-9, (
        f"kappa_ceiling came back {c.kappa_ceiling!r}; expected sqrt(0.64) = 0.8. 0.64 is the "
        "reviewers' kappa itself — what a system as good as reviewer B scores against A. A "
        "perfect system shares c with A, not c x c.")
    assert abs(c.accuracy_ceiling - 0.9) < 1e-9, (
        f"accuracy_ceiling came back {c.accuracy_ceiling!r}; expected 0.5 + 0.8 x (1 - 0.5) = 0.9"
        ". 0.82 would be the reviewers' raw agreement, which is a floor for a perfect system, "
        "not its ceiling.")
    worse = harness_ceiling(["fix", "accept"] * 5, ["accept", "fix"] * 5)
    assert worse.kappa_ceiling == 0.0 and abs(worse.accuracy_ceiling - 0.5) < 1e-9, (
        f"got {worse}: kappa is -1, so there is no signal — kappa_ceiling 0.0 and the accuracy "
        "ceiling falls to chance agreement, 0.5. Do not take the root of a negative number.")
    try:
        harness_ceiling(["accept"] * 4, ["accept"] * 4)
    except ValueError:
        pass
    else:
        raise AssertionError("one label for everything gives a nan kappa: raise ValueError")
    print("exercise 5 looks right")


_try("exercise 5", _check_ceiling)

Now the measurement itself. First the ceiling from your audit kappa, beside the agreement
between the truth and each reviewer — which only a simulation can compute, and which is what
the ceiling claims to estimate. Then the same ceiling in the harness's own units: gold for the
whole corpus written the way this lesson's queue works — A sees the machine's value and
accepts, fixes or clears it — and a perfect extractor scored against it.

In [ ]:
def gold_written_by(verdicts: Sequence[str]) -> list[dict]:
    """Module 1's records, with each cell's gold replaced by what a reviewer's verdict made of it."""
    verdict_of = dict(zip(AUDIT_CELLS, verdicts))
    out = []
    for record in RECORDS:
        gold = {f: apply_verdict(record["pred"][f], record["gold"][f], verdict_of[(record["doc_id"], f)])
                for f in SCHEMA}
        out.append({"doc_id": record["doc_id"], "gold": gold, "pred": dict(record["pred"]),
                    "conf": dict(record["conf"])})
    return out


def _show_ceiling() -> None:
    c = harness_ceiling(AUDIT_A, AUDIT_B)
    print(f"from the audit: kappa {c.kappa:.3f}  ->  ceiling kappa {c.kappa_ceiling:.3f}, "
          f"raw agreement {c.accuracy_ceiling:.3f}")
    for name, labels in (("A", AUDIT_A), ("B", AUDIT_B)):
        print(f"truth against {name} (simulation only): kappa {cohens_kappa(AUDIT_TRUTH, labels):.3f}, "
              f"raw agreement {raw_agreement(AUDIT_TRUTH, labels):.3f}")

    gold_a = gold_written_by(AUDIT_A)
    perfect = [dict(r, pred=dict(t["gold"])) for r, t in zip(gold_a, RECORDS)]
    perfect_f1, machine_f1 = macro_f1(perfect, "normalised"), macro_f1(gold_a, "normalised")
    print(f"\nin the harness's own units, gold written by A:")
    print(f"  a PERFECT extractor         macro F1 {perfect_f1:.3f}")
    print(f"  module 1's extractor        macro F1 {machine_f1:.3f}   (against the truth: {MACHINE_F1:.3f})")
    print(f"  headroom the harness shows  {perfect_f1 - machine_f1:.3f}   (really: {1 - MACHINE_F1:.3f})")
    print("\nThe ceiling is set by your reviewers' agreement, not 1.0. Past it, a better extractor")
    print("and a better copy of your annotators' mistakes produce the same number.")


_try("the ceiling", _show_ceiling, needs=("exercise 1", "exercise 2", "exercise 5"))

## 11. The artefact: a review policy with a measured ceiling

One card, every line computed above: the queue, the order, the rota, the escalation rules,
what they cost in breaches, what they buy in quality, and the ceiling on all of it. This is the
page a model-risk reviewer or an oversight lead signs — or sends back.

In [ ]:
def _show_policy_card() -> None:
    results = sweep_shifts()
    chosen = choose_shift(results)
    if chosen is None:
        print(f"no shift length meets BREACH_TARGET = {BREACH_TARGET}; there is no policy to sign")
        return
    ceiling = harness_ceiling(AUDIT_A, AUDIT_B)
    print("REVIEW POLICY · field-extraction review queue")
    print(f"  queue            module 1's review_queue: {len(DAY)} lowest-confidence cells, "
          f"{len(RECORDS)} documents")
    print(f"  order            earliest deadline first; SLA {SLA_MINUTES['money']} min money, "
          f"{SLA_MINUTES['id']} min other")
    print(f"  tier 1           reviewer A, shifts of {chosen.shift_items} items, "
          f"{BREAK_MINUTES}-minute breaks, {chosen.per_hour:.1f} items per hour at the desk")
    print(f"  tier 2           reviewer S: four eyes on {', '.join(sorted(FOUR_EYES_TYPES))}; "
          f"accept below confidence {RUBBER_STAMP_CONF} escalated")
    print(f"  escalated        {chosen.escalated:.1f} of {len(DAY)} cells a day "
          f"({100 * chosen.escalated / len(DAY):.1f}%)")
    print(f"  SLA breach rate  {chosen.breach_rate:.3f} mean, {chosen.breach_worst:.3f} worst day, "
          f"over {N_DAYS} days (target {BREACH_TARGET})")
    print(f"  still wrong      {chosen.still_wrong:.1f} of {len(DAY)} routed cells a day")
    print(f"  macro F1         {chosen.macro_f1:.3f} against the truth "
          f"(perfect reviewer {PERFECT_REVIEW_F1:.3f})")
    print(f"  agreement        raw {raw_agreement(AUDIT_A, AUDIT_B):.3f}, kappa {ceiling.kappa:.3f} "
          f"(A and B on {len(AUDIT_A)} cells)")
    print(f"  quality ceiling  kappa {ceiling.kappa_ceiling:.3f}, raw agreement "
          f"{ceiling.accuracy_ceiling:.3f}: the most a perfect system can score against gold")
    print("                   these reviewers write, assuming their mistakes are independent")


_try("policy card", _show_policy_card, needs=("exercise 1", "exercise 2", "exercise 3",
                                               "exercise 4", "exercise 5"))

## 12. Common mistakes

- **Reporting raw agreement as reliability.** On a label mix dominated by `accept`, a reviewer
  who never looked scores high raw agreement and zero kappa. Section 5 measured both.
- **Pairing label frequencies by position.** Chance agreement is a sum matched BY LABEL over
  the union of both reviewers' labels; B never says `clear`, and a sorted-and-zipped version
  silently pairs the wrong labels.
- **Calling one-label agreement perfect.** Two reviewers who accepted everything agree on every
  item and have measured nothing. Kappa is undefined; say so.
- **Sorting the queue once by deadline.** It serves cells before they arrive, and its breach
  rate is fiction. A queue is a clock.
- **Using module 1's confidence order for timing.** It decides what is worth a human's look,
  not what is due first.
- **Escalating `fix` but not `clear`, or answering `False` for an unknown verdict.** Both send a
  money change nobody checked to the payment run.
- **Treating a verdict as the truth.** Module 1's `apply_reviews` did. Every reviewer here errs,
  more so late in a shift.
- **Reading the ceiling as 1.0, or as kappa itself.** A perfect system shares √κ with one
  reviewer. And the root assumes independent mistakes: reviewers who make the SAME mistakes
  agree more than their accuracy deserves, and their ceiling reads high. Run the next cell.

In [ ]:
def _show_shared_mistakes() -> None:
    twin = REVIEWER_B._replace(name="B'", seed=REVIEWER_A.seed)   # same dice as A: shared mistakes
    audit_twin = run_audit(twin)
    for name, labels in (("A and B, independent mistakes", AUDIT_B),
                         ("A and B', shared mistakes", audit_twin)):
        c = harness_ceiling(AUDIT_A, labels)
        print(f"{name:32s} kappa {c.kappa:.3f} -> ceiling kappa {c.kappa_ceiling:.3f}")
    print(f"{'truth against A (simulation only)':32s} kappa {cohens_kappa(AUDIT_TRUTH, AUDIT_A):.3f}")
    print("\nA's labels did not change, so neither did the truth about them. Only the second")
    print("reviewer did — and a pair that errs together reports a ceiling A never reaches.")


_try("shared mistakes", _show_shared_mistakes, needs=("exercise 1", "exercise 2", "exercise 5"))

## 13. Self-check

1. Two reviewers agree on most cells, but their kappa is modest. The right reading is:
   - (a) kappa has been computed wrongly, since it cannot be far below raw agreement
   - (b) much of their agreement is what their label mix would produce by chance
   - (c) both reviewers are as accurate as their raw agreement

2. Reviewer B never uses `clear`. You compute chance agreement by sorting each reviewer's
   distinct labels, counting them, and multiplying the two lists element by element. You get:
   - (a) the right answer, because a label B never used contributes zero anyway
   - (b) the right answer whenever both lists come out the same length
   - (c) a wrong pairing whenever the two reviewers' label sets differ: an error if you are
         lucky, and a plausible number that raises nothing if you are not

3. A colleague's queue sorts every item by deadline once and serves that list in order, one
   item after another. Its breach rate looks respectable. What is wrong with it?
   - (a) it starts cells before they have arrived, so its schedule could never have happened
         and its breach rate describes nothing
   - (b) nothing: sorting once is simply more efficient than choosing at every step
   - (c) it breaks ties differently from yours

4. At the same number of escalations, the rules catch more tier-1 mistakes than the random
   control. That comparison measures:
   - (a) how much more accurate reviewer S is than reviewer A
   - (b) nothing, since both send the same number of cells to tier 2
   - (c) the value of the policy — where it spends senior time — with the senior held fixed

5. Your reviewers' kappa is 0.64. A vendor's extractor scores kappa 0.90 against gold one of
   your reviewers wrote. The right response is:
   - (a) suspicion: that is above the ceiling of about 0.8, so either the extractor is
         reproducing your annotator's mistakes or the gold was not made the way you think
   - (b) adopt it: it beats your own reviewers
   - (c) nothing can be concluded from kappa values below 1.0

Answers come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

You now own the human half of the harness: an agreement table, raw agreement and kappa that
survive every edge case, a queue that is a clock rather than a sort, an escalation policy that
beats random at the same cost, a rota chosen against a breach target, and the ceiling that
your reviewers' agreement puts on every number module 1's harness reports. Module 9 prices the
human tier this lesson measured; the capstone asks you to put the policy card, and its
ceiling, in front of a validator.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_agreement),
                              ("exercise 2", _check_kappa),
                              ("exercise 3", _check_schedule),
                              ("exercise 4", _check_escalate),
                              ("exercise 5", _check_ceiling)):
            _try(_name, _check)
    _progress_board()
    _wall = time.perf_counter() - _LESSON_T0
    # Whole seconds: two machines disagree at the first decimal, and that is noise, not a result.
    print("\nlesson wall time so far: " + ("under a second" if _wall < 1 else f"{_wall:.0f}s"))
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.